In [1]:
vel = True


import yaml
import os
import subprocess
import pandas as pd
import numpy as np
from itertools import product
import glob

def create_param_string(model_params):
    param_strings = []
    for key, val in model_params.items():
        param_strings.append(f"{key}={val}")
    return "_".join(param_strings)

def update_yaml_files_and_save_params(train_yaml_file, model_yaml_file, lr_value, model_params):
    """
    Update two YAML files with specific parameters and return a parameter info string
    """
    # Load and update the train YAML file
    with open(train_yaml_file, 'r') as file:
        train_data = yaml.safe_load(file)
    
    # Update learning rate in train config
    train_data['learning_rate'] = lr_value
    
    # Save the updated train YAML
    with open(train_yaml_file, 'w') as file:
        yaml.dump(train_data, file, default_flow_style=False)
    
    # Load and update the model YAML file
    with open(model_yaml_file, 'r') as file:
        model_data = yaml.safe_load(file)
    
    # Update model parameters
    for key, value in model_params.items():
        model_data[key] = value
    
    # Save the updated model YAML
    with open(model_yaml_file, 'w') as file:
        yaml.dump(model_data, file, default_flow_style=False)
    
    # Create a parameter info string combining all parameters
    param_info = f"lr={lr_value}_" + create_param_string(model_params)
    
    return param_info

def run_training_script(data_config, model_config, train_config, dataset, param_info):
    """
    Run the training script with the specified arguments
    """
    if vel:
        new_pre = "vel_" + param_info
    else:
        new_pre = param_info
    cmd = [
        "python", "/home/bsb2144/daart/examples/fit_models_subsample_loop.py",
        "--pre", new_pre,  # Use param_info as the --pre argument
        "--fit_tcn",
        "--frac", "1",
        "--dataset", dataset,
        "--n_samples", "1",
        "--data", data_config,
        "--model", model_config,
        "--train", train_config
    ]
    
    print(f"Running: {' '.join(cmd)}")
    print(f"With parameters: {param_info}")
    
    # Use subprocess.run to execute the command
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Error running script: {result.stderr}")
        return False
    else:
        print("Script executed successfully")
        return True

def evaluate_directory(dir_name, n_vs):
    """
    Evaluate a directory by finding the minimum val_loss where dataset=-1 in each version
    and averaging them.
    
    Parameters:
    dir_name (str): Directory name to evaluate
    
    Returns:
    float: Average of minimum val_loss values across versions
    """
    min_val_losses = []
    
    # Process each version subdirectory
    for version in range(n_vs):  # version_0 through version_4
        version_dir = os.path.join(dir_name, f"version_{version}")
        metrics_path = os.path.join(version_dir, "metrics.csv")
        
        try:
            # Read the metrics CSV file
            df = pd.read_csv(metrics_path)
            
            # Filter rows where dataset=-1 and find minimum val_loss
            filtered_df = df[df['dataset'] == -1]
            if not filtered_df.empty:
                min_val_loss = filtered_df['val_loss'].min()
                min_val_losses.append(min_val_loss)
                #print(f"Min val_loss for {version_dir}: {min_val_loss}")
        
        except Exception as e:
            print(f"Error processing {metrics_path}: {e}")
    
    # Calculate average if we have any valid values
    if min_val_losses:
        avg_score = np.mean(min_val_losses)
        print(f"Average min val_loss for {dir_name}: {avg_score}")
        return avg_score
    else:
        print(f"No valid data found for {dir_name}")
        return float('inf')  # Return infinity if no valid data

def get_hyperparameters(dir_name):
    """
    Get hyperparameters from the version_0/hparams.yaml file
    
    Parameters:
    dir_name (str): Directory name
    
    Returns:
    dict: Hyperparameters from the YAML file
    """
    hparams_path = os.path.join(dir_name, "version_0", "hparams.yaml")
    
    try:
        with open(hparams_path, 'r') as file:
            hparams = yaml.safe_load(file)
            return hparams
    except Exception as e:
        print(f"Error reading {hparams_path}: {e}")
        return None

def grid_search_hyperparameters(train_yaml_file, model_yaml_file, data_config, model_config, train_config, dataset, base_dir, n_vs):
    """
    Perform grid search over hyperparameter combinations, run training script, and evaluate results
    """
    # Define hyperparameter values to search
    #{"input_type": "features-vit_m", "subdir": "lr=0.0001_dropout=0.1_n_hid_units=32_n_lags=16_n_hid_layers=2-68-good_sample-0_",
#     
    # lr=1e-05_dropout=0.1_n_hid_units=32_n_lags=16
    learning_rates = [0.0001, 0.00001]#, 0.001]
    dropout_rates = [0.1]
    hidden_units = [32, 64]
    n_lags = [4, 8, 16]
    n_hid_layers = [2]


    # learning_rates = [0.0001]
    # dropout_rates = [0.1]
    # hidden_units = [32]
    # n_lags = [16]
    # n_hid_layers = [2]
    
    # Store all parameter info strings
    param_info_list = []
    
    # Loop through all combinations
    for lr, drop, hidden,lag, n_layer in product(learning_rates, dropout_rates, hidden_units, n_lags, n_hid_layers):
        # Create model parameters dictionary
        model_params = {
            'dropout': drop,
            'n_hid_units': hidden,
            "n_lags": lag,
            "n_hid_layers": n_layer
        }
        
        # Update YAML files and get parameter info
        param_info = update_yaml_files_and_save_params(
            train_yaml_file, 
            model_yaml_file, 
            lr, 
            model_params
        )
        param_info_list.append(param_info)
        
        # Run the training script with updated configs
        success = run_training_script(data_config, model_config, train_config, dataset, param_info)
        if not success:
            print(f"Skipping evaluation for {param_info} due to training failure")
    
    print(f"Completed {len(param_info_list)} parameter combinations")
    
    # Evaluate directories after all runs complete
    evaluate_and_find_best_model(param_info_list, base_dir, n_vs)

def evaluate_and_find_best_model(param_info_list, base_dir, n_vs):
    """
    Evaluate all directories and find the one with the smallest average val_loss
    
    Parameters:
    param_info_list (list): List of parameter info strings (directory names)
    """
    # Dictionary to store directory scores
    scores = {}
    # Evaluate each directory
    
    for param_info in param_info_list:
        if vel:
            new_pre = "vel_" + param_info
        else:
            new_pre = param_info
        score = evaluate_directory(base_dir.format(param_info), n_vs)
        scores[param_info] = score
    
    # Find directory with the smallest score
    if scores:
        best_dir = min(scores, key=scores.get)
        best_score = scores[best_dir]
        
        print("\n" + "="*50)
        print(f"Best directory: {best_dir}")
        print(f"Best average val_loss: {best_score}")
        
        return best_dir, best_score
    else:
        print("No valid directories to evaluate")
        return None, None


In [ ]:
# Configuration files
data_config = "/home/bsb2144/daart_utils/configs/data_c4.yaml"
model_config = "/home/bsb2144/daart_utils/configs/model_c4.yaml"
train_config = "/home/bsb2144/daart_utils/configs/train_c4.yaml"

# base model dir
#base_dir = "/home/bsb2144/daart/results_daart/ibl/multi-22/dtcn/{}-18-good_sample-0_markers3"
base_dir = "/home/bsb2144/daart/results_daart/calms21/multi-0/dtcn/{}-68-good_sample-0_features-vit_mae"
# Dataset name
dataset = "calms21"

# number of trial versions
n_vs=3


# YAML files to update with hyperparameters
train_yaml_file = train_config  # Contains learning_rate
model_yaml_file = model_config  # Contains dropout and n_hidden_units

grid_search_hyperparameters(
    train_yaml_file, 
    model_yaml_file, 
    data_config, 
    model_config, 
    train_config, 
    dataset,
    base_dir,
    n_vs
)

Running: python /home/bsb2144/daart/examples/fit_models_subsample_loop.py --pre vel_lr=0.0001_dropout=0.1_n_hid_units=32_n_lags=4_n_hid_layers=2 --fit_tcn --frac 1 --dataset calms21 --n_samples 1 --data /home/bsb2144/daart_utils/configs/data_c4.yaml --model /home/bsb2144/daart_utils/configs/model_c4.yaml --train /home/bsb2144/daart_utils/configs/train_c4.yaml
With parameters: lr=0.0001_dropout=0.1_n_hid_units=32_n_lags=4_n_hid_layers=2


In [10]:
print(16/4.3, 20/5.4)

3.7209302325581395 3.7037037037037033


In [4]:
import multiprocessing
print(f"CPU cores available: {multiprocessing.cpu_count()}")
import torch
print(f"Detected GPUs: {torch.cuda.device_count()}")


CPU cores available: 256
Detected GPUs: 1
